# Quishing Technical Demonstration
## Q14_04_DEM01 — From a Stored QR Image to an Explained Outcome

This notebook presents five representative cases from our quishing study. We show a legitimate QR, a detected phishing QR, a successful path-encoding evasion against the earlier model, the same stored attack after model hardening, and an unreadable QR that triggers abstention.

We keep the full experimental history in notebook `02`. This notebook reads saved research artifacts without importing or executing that notebook, fitting a model, recalibrating a threshold, or contacting a destination.

<font color="violet">This is an evidence-backed demonstration. Scores and classifier verdicts are loaded from saved research results. QR decoding is performed again when these cells run. Autonomous classifier inference and fresh-environment reproduction belong to Q14_04_QA01.</font>

**Reading convention:** green means a correct observed outcome, red identifies a verified weakness, and violet highlights a methodological condition. A legitimate prediction is not a guarantee that an address is safe.

### 1. Define the Demonstration Scenario

| Case | What we show | Evidence version |
|---|---|---|
| 1 | A legitimate QR correctly classified | R2 hostname-and-path normalization |
| 2 | A phishing QR correctly detected | R2 |
| 3 | A stored encoded-path QR classified as legitimate | R1 hostname-only normalization |
| 4 | The same attack included in the detected replay cohort | R2, EXP08 seed 42 |
| 5 | Blur makes a QR unreadable and produces abstention | Decoder check; historical R1 baseline |

We use **seed 42 by a fixed presentation convention**, not because it outperformed another seed. The three repetitions produced the same aggregate results.

Open this notebook in the experiment directory or its `notebooks` subdirectory. Keep the existing `outputs/quishing02` tree available. Run the cells in order. If several runs exist, explicitly select the intended directory in the configuration cell.

In [ ]:
# Import the tools used to read and present existing evidence.
from pathlib import Path
from io import BytesIO
from html import escape
import hashlib
import json
import re
import platform
import numpy as np
import pandas as pd
from PIL import Image
from IPython.display import display, HTML

print(f"Python {platform.python_version()} — evidence replay, without model fitting.")

### Locate the Saved Research Artifacts

We resolve one experiment root and one run per evidence family. Directory discovery never selects the newest run silently. If several directories match, choose one explicitly below.

The EXP08 directory is pinned to the run recorded in the supplied notebook. The path-attack directory contains both the original attack PNGs and the corrected clean-QR verification. All paths remain local.

In [ ]:
# Change PROJECT_ROOT only if this notebook is stored outside the project.
PROJECT_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                     if (p / "outputs/quishing02").is_dir()), Path.cwd())
OUTPUT_ROOT = PROJECT_ROOT / "outputs/quishing02"
EXP08_OVERRIDE = None  # Optional absolute or project-relative directory.
PATH_SEARCH_OVERRIDE = None  # Directory ending in schemeless_search.
CORRECTED_OVERRIDE = None  # Optional corrected_search_* directory.
SEED = 42
PINNED_EXP08_NAME = "exp08_repeated_training_20261121T170229937001Z"

# Require an explicit choice when more than one run is available.
def select_directory(override, candidates, label):
    if override is not None:
        chosen = Path(override)
        chosen = chosen if chosen.is_absolute() else PROJECT_ROOT / chosen
        if not chosen.is_dir():
            raise FileNotFoundError(f"{label}: directory not found: {chosen}")
        return chosen.resolve()
    choices = sorted(set(p.resolve() for p in candidates))
    if len(choices) != 1:
        listed = "\n".join(map(str, choices)) or "No matching directory."
        raise RuntimeError(f"Select {label} explicitly in the configuration cell.\n{listed}")
    return choices[0]

exp08_dir = select_directory(EXP08_OVERRIDE,
    OUTPUT_ROOT.rglob(PINNED_EXP08_NAME), "EXP08_OVERRIDE")
normalized_dir = exp08_dir.parent
path_search_dir = select_directory(PATH_SEARCH_OVERRIDE,
    [p for p in normalized_dir.glob("exp13_path_encoding_*/schemeless_search")
     if (p / "stored_qr_verification/details.csv").is_file()],
    "PATH_SEARCH_OVERRIDE")
corrected_dir = select_directory(CORRECTED_OVERRIDE,
    [p for p in path_search_dir.glob("corrected_search_*")
     if (p / "clean_qr_verification/results.csv").is_file()],
    "corrected path-search run (remove ambiguity before continuing)")

print(f"Research outputs: {OUTPUT_ROOT}\nEXP08: {exp08_dir.name}"
      f"\nAttack run: {path_search_dir.parent.name}\nCorrected run: {corrected_dir.name}")

#### Output Explanation

The printed paths identify the runs used throughout the demonstration. An ambiguity is a configuration issue, not an experiment failure. We stop rather than mix evidence from different runs.

The files must already exist on Cube-AI. Downloading this notebook alone does not download the dataset, checkpoints or saved QR images.

### 2. Check Evidence Integrity and Load the Results

We verify the SHA-256 hashes recorded by EXP08 before reading its tables. These checks detect changes relative to the saved manifest; they do not independently authenticate the original experiment.

We do not load joblib models here. This keeps notebook-defined Python functions from becoming a hidden execution dependency.

In [ ]:
def read_json(path):
    return json.loads(path.read_text(encoding="utf-8"))

# Check every evidence file listed in the research manifest.
manifest = read_json(exp08_dir / "evidence_manifest.json")
for item in manifest["evidence"]:
    file = exp08_dir / item["file"]
    if not file.is_file():
        raise FileNotFoundError(file)
    actual = hashlib.sha256(file.read_bytes()).hexdigest()
    if actual != item["sha256"]:
        raise ValueError(f"Evidence hash mismatch: {file.name}")

protocol = read_json(exp08_dir / "protocol.json")
path_protocol = read_json(path_search_dir / "protocol.json")
clean = pd.read_csv(corrected_dir / "clean_qr_verification/results.csv")
attacks = pd.read_csv(path_search_dir / "stored_qr_verification/details.csv")
pairs = pd.read_csv(path_search_dir / "source_results.csv")
training = pd.read_csv(exp08_dir / "training_summary.csv")
predictions = pd.read_csv(exp08_dir / "predictions.csv")
checks = pd.read_csv(exp08_dir / "path_attack_checks.csv")
decode_checks = pd.read_csv(exp08_dir / "stored_attack_decode_checks.csv")
ci = pd.read_csv(exp08_dir / "domain_cluster_confidence_intervals.csv")
membership = pd.read_csv(exp08_dir / "subset_membership.csv")
print(f"EXP08 integrity checks passed for {len(manifest['evidence'])} evidence files.")

#### Output Explanation

The tables serve different purposes. `clean` contains recorded R2 image-pipeline outcomes. `attacks` contains recorded R1 outcomes for the saved attack PNGs. EXP08 supplies the final seed-specific thresholds, original-address scores and attack-regression checks.

<font color="violet">We do not relabel an earlier-version result as a final-version measurement. R1 and R2 scores use their own calibrated thresholds.</font>

In [ ]:
# Parse CSV booleans explicitly; the string "False" must never become True.
def bool_column(frame, column):
    values = frame[column].astype(str).str.lower().str.strip()
    if not values.isin(["true", "false", "1", "0"]).all():
        raise ValueError(f"Invalid boolean values in {column}")
    return values.isin(["true", "1"])

for column in ["accepted_decode", "exact_reference_payload", "accepted_payload_mismatch"]:
    clean[column] = bool_column(clean, column)
for column in ["verified_conditional_evasion", "accepted_decode", "exact_candidate_payload"]:
    attacks[column] = bool_column(attacks, column)
for column in ["accepted_decode", "exact_candidate_payload"]:
    decode_checks[column] = bool_column(decode_checks, column)

for name, frame in [("clean", clean), ("attacks", attacks), ("pairs", pairs)]:
    if frame["image_index"].duplicated().any():
        raise ValueError(f"Duplicate source IDs in {name}")

seed_validation = predictions.loc[
    predictions["seed"].eq(SEED) & predictions["subset"].eq("Original validation")
].set_index("image_index", verify_integrity=True)
payload_hashes = membership.loc[membership["subset"].eq("validation")].set_index(
    "image_index", verify_integrity=True)["payload_sha256"]
seed_check = checks.set_index("seed", verify_integrity=True).loc[SEED]
threshold_rows = training.loc[training["seed"].eq(SEED), "threshold"].unique()
assert len(threshold_rows) == 1
r2_threshold = float(threshold_rows[0])
r1_threshold = float(path_protocol["threshold"])

# Validate the exact cohort used to explain the post-correction example.
verified = attacks.loc[attacks["verified_conditional_evasion"]].copy()
assert len(verified) == int(seed_check["stored_attack_PNGs"])
assert int(seed_check["exact_stored_candidates_detected"]) == len(verified)
assert int(seed_check["exact_stored_candidates_missed"]) == 0
assert int(seed_check["retained_pair_evasions"]) == 0
assert float(seed_check["maximum_pair_score_difference"]) == 0.0
stored_ids = set(decode_checks.loc[
    decode_checks["accepted_decode"] & decode_checks["exact_candidate_payload"], "image_index"])
assert set(verified["image_index"]) == stored_ids
assert set(verified["image_index"]).issubset(set(pairs["image_index"]))
assert len(clean) == 1956 and set(clean["image_index"]) == set(seed_validation.index)
print(f"R1 threshold: {r1_threshold:.9f} | R2 seed {SEED}: {r2_threshold:.9f}"
      f"\nVerified stored attack cohort: {len(verified)} sources.")

### 3. Prepare Image Reading and Clear Result Displays

We read actual PNG files from disk and use the same decoder acceptance rule as the study: accept one distinct nonempty payload across OpenCV and ZXing. If decoded texts disagree, or neither decoder recovers text, abstain.

Displayed payloads are escaped and defanged text. No payload is opened. Reference text is used only to audit recovery and is never supplied as a fallback to a decoder or classifier.

In [ ]:
import cv2
import zxingcpp

# Decode display-polarity PNG bytes with the tested 69 x 69 image convention.
def decode_png(png_bytes):
    image = cv2.imdecode(np.frombuffer(png_bytes, np.uint8), cv2.IMREAD_GRAYSCALE)
    if image is None or image.shape != (69, 69):
        raise ValueError("Expected a readable 69 x 69 grayscale PNG.")
    qr = cv2.copyMakeBorder(image, 4, 4, 4, 4, cv2.BORDER_CONSTANT, value=255)
    qr = cv2.resize(qr, None, fx=3, fy=3, interpolation=cv2.INTER_NEAREST)
    try:
        cv_text, _, _ = cv2.QRCodeDetector().detectAndDecode(qr)
    except (cv2.error, UnicodeDecodeError):
        cv_text = ""
    try:
        results = zxingcpp.read_barcodes(qr, formats=zxingcpp.BarcodeFormat.QRCode,
            try_rotate=True, try_downscale=True, try_invert=True,
            text_mode=zxingcpp.TextMode.Plain)
        zx_values = {r.text for r in results if r.valid and r.text}
    except (RuntimeError, UnicodeDecodeError):
        zx_values = set()
    values = zx_values | ({cv_text} if cv_text else set())
    return {"text": next(iter(values)) if len(values) == 1 else "",
            "accepted": len(values) == 1, "conflict": len(values) > 1,
            "opencv": cv_text, "zxing": sorted(zx_values)}

# Rebase an artifact path when the project has been moved to another machine.
def artifact_path(saved_path):
    raw = Path(str(saved_path))
    if raw.is_file():
        return raw
    parts = raw.parts
    if "outputs" in parts:
        rebased = PROJECT_ROOT.joinpath(*parts[parts.index("outputs"):])
        if rebased.is_file():
            return rebased
    raise FileNotFoundError(f"Stored artifact not found: {saved_path}")

def display_case(title, png_bytes, rows, tone="green"):
    color = {"green": "#287443", "red": "#B3261E", "violet": "#7343A5"}[tone]
    display(HTML(f'<h3 style="color:{color}">{escape(title)}</h3>'))
    display(Image.open(BytesIO(png_bytes)).resize((276, 276), Image.Resampling.NEAREST))
    safe = pd.DataFrame(rows, columns=["Field", "Value"])
    display(HTML(safe.to_html(index=False, escape=True)))

print("Decoder and display helpers ready. Classifier inference is not called.")

#### Output Explanation

The image is enlarged only for presentation. Decoding uses the stored 69 × 69 pixels, with the research border and resize convention. A current decode failure is shown explicitly rather than replaced by the expected text.

Each result panel identifies whether a value was **decoded now**, **recorded in research**, or **inferred from a checked cohort result**. This distinction prevents an evidence replay from being mistaken for a fresh model evaluation.

### Select Representative Examples Before Presenting Them

We deterministically choose the lowest source ID among correct legitimate outcomes and the lowest source ID among correct phishing outcomes. For the attack, we choose the largest recorded R1 score drop, breaking ties by source ID.

These are deliberately selected teaching examples. They are not a new random evaluation cohort, and their success rate must not replace the complete validation results.

In [ ]:
accepted_clean = clean.loc[clean["accepted_decode"] & clean["exact_reference_payload"]]
legitimate_example = accepted_clean.loc[
    accepted_clean["label"].eq(0) & accepted_clean["prediction"].eq(0)
].sort_values("image_index").iloc[0]
phishing_example = accepted_clean.loc[
    accepted_clean["label"].eq(1) & accepted_clean["prediction"].eq(1)
].sort_values("image_index").iloc[0]
attack_example = verified.assign(
    score_drop=lambda f: f["original_score"] - f["stored_score"]
).sort_values(["score_drop", "image_index"], ascending=[False, True]).iloc[0]
attack_id = int(attack_example["image_index"])
attack_pair = pairs.set_index("image_index").loc[attack_id]
assert attack_example["candidate_payload"] == attack_pair["candidate_payload"]
assert attack_example["original_payload"] == attack_pair["original_payload"]
selected = pd.DataFrame([
    {"case": "1 Legitimate", "image_index": int(legitimate_example["image_index"])},
    {"case": "2 Phishing", "image_index": int(phishing_example["image_index"])},
    {"case": "3 and 4 Same stored attack", "image_index": attack_id},
    {"case": "5 Blur of case 2", "image_index": int(phishing_example["image_index"])}])
display(selected)

### Case 1 — A Legitimate QR Correctly Classified

We read a clean stored QR, decode it again, and display its recorded R2 score and verdict. The score is a classifier output, not a calibrated probability or a safety guarantee.

In [ ]:
def show_clean_example(row, title):
    image_id = int(row["image_index"])
    path = corrected_dir / "clean_qr_verification" / f"{image_id}.png"
    png = path.read_bytes()
    decoded = decode_png(png)
    decoded["exact_reference"] = decoded["accepted"] and hashlib.sha256(
        decoded["text"].encode("utf-8")).hexdigest() == payload_hashes.loc[image_id]
    saved = seed_validation.loc[image_id]
    assert int(saved["prediction"]) == int(row["prediction"])
    assert int(saved["label"]) == int(row["label"])
    # Clean-pipeline and seed-42 scores may differ slightly between independent fits.
    assert (float(row["score"]) >= float(protocol["reference_threshold"])) == bool(row["prediction"])
    payload = decoded["text"].replace(".", "[.]") if decoded["accepted"] else "No accepted payload"
    display_case(title, png, [
        ("Source ID", str(image_id)), ("Model evidence", f"R2 EXP08 seed {SEED}"),
        ("Decode now", "Accepted" if decoded["accepted"] else "Abstention"),
        ("Exact reference recovered now", str(decoded["exact_reference"])),
        ("Payload decoded now, defanged", payload),
        ("Saved score", f"{float(saved['score']):.9f}"),
        ("Saved threshold", f"{float(saved['threshold']):.9f}"),
        ("Recorded verdict", "Phishing" if int(saved["prediction"]) else "Legitimate"),
        ("Provenance", "Saved prediction; no classifier execution in this notebook")])
    return png, decoded

legitimate_png, legitimate_decode = show_clean_example(
    legitimate_example, "Case 1 — Correct legitimate outcome")
print("Recorded classification displayed; current decoding is reported separately.")

#### Output Explanation

The recorded legitimate verdict corresponds to a labelled legitimate example. If the current decoder abstains, the current pipeline would stop before classification, even though a historical score is available. The display keeps that distinction visible.

A green heading identifies the selected historical outcome; it does not certify a destination.

### Case 2 — A Phishing QR Correctly Detected

We apply the same presentation to a labelled phishing example. Detection means its saved score reaches or exceeds its own saved threshold.

In [ ]:
phishing_png, phishing_decode = show_clean_example(
    phishing_example, "Case 2 — Correct phishing detection")
print("One selected example illustrates detection; full-cohort recall is reported later.")

#### Output Explanation

A phishing verdict on this example illustrates expected behaviour. It does not imply every phishing QR is detected: the final clean cohort still contains 102 missed phishing examples. The representative example and the complete evaluation answer different questions.

### Case 3 — A Verified Path-Encoding Evasion Before Correction

The attack replaced eligible ASCII unreserved path characters with `%HH` triplets while preserving the raw hostname, path separators, query and fragment. For example, `/login` can become `/%6Cogin`.

We use the actual retained PNG and the recorded R1 scores. A fresh decode must recover the exact candidate text before this run can confirm payload recovery.

<font color="violet">Destination equivalence is conditional: scheme-less strings are interpreted as hostname-plus-path web addresses, and the changed path has the same unreserved-decoded form. No destination was contacted to verify browser or server behaviour.</font>

In [ ]:
# Check the exact transformation rule without visiting an address.
UNRESERVED = frozenset("ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz0123456789-._~")
def split_raw_address(text):
    start = 2 if text.startswith("//") else 0
    while start < len(text) and text[start] not in "/?#":
        start += 1
    end = start
    while end < len(text) and text[end] not in "?#":
        end += 1
    return text[:start], text[start:end], text[end:]

def unreserved_form(path):
    return re.sub(r"%[0-9A-Fa-f]{2}", lambda m:
        chr(int(m.group()[1:], 16)) if chr(int(m.group()[1:], 16)) in UNRESERVED
        else m.group(), path)

original = str(attack_pair["original_payload"])
candidate = str(attack_pair["candidate_payload"])
authority, original_path, suffix = split_raw_address(original)
c_authority, encoded_path, c_suffix = split_raw_address(candidate)
assert (authority, suffix) == (c_authority, c_suffix)
assert unreserved_form(original_path) == unreserved_form(encoded_path)
attack_png_path = artifact_path(attack_example["stored_png"])
attack_png = attack_png_path.read_bytes()
attack_decode = decode_png(attack_png)
current_exact = attack_decode["accepted"] and attack_decode["text"] == candidate
assert float(attack_example["original_score"]) >= r1_threshold
assert float(attack_example["stored_score"]) < r1_threshold

display_case("Case 3 — Recorded R1 evasion", attack_png, [
    ("Source ID", str(attack_id)), ("Model evidence", "R1 hostname-only normalization"),
    ("Original path", original_path), ("Encoded path", encoded_path),
    ("Equivalence rule", "Same unreserved-decoded path; raw text outside path unchanged"),
    ("Original recorded score", f"{float(attack_example['original_score']):.9f}"),
    ("Stored candidate recorded score", f"{float(attack_example['stored_score']):.9f}"),
    ("R1 threshold", f"{r1_threshold:.9f}"),
    ("Decode now", "Exact candidate recovered" if current_exact else "Recovery not confirmed"),
    ("Payload decoded now, defanged", attack_decode["text"].replace(".", "[.]")),
    ("Historical verdict", "Legitimate — verified conditional evasion")], "red")
print("A new decode check and saved R1 classifier results are shown separately.")

#### Output Explanation

The original score was above the R1 threshold, while the candidate score was below it. The research run saved the candidate as a PNG, reloaded it and recovered the exact candidate payload. That established a classifier evasion within the stated equivalence assumptions.

**69 of 755 eligible, initially detected sources** produced verified conditional evasions (9.14%). This rate is conditional on attack eligibility and initial detection. It is neither a rate over all 972 phishing sources nor a pixel-distortion result.

### Case 4 — The Same Stored Attack After Model Hardening

We reuse exactly the same PNG bytes. R2 normalizes supported hostname case and percent-encoded ASCII unreserved path characters before feature extraction. Reserved encodings, queries and fragments are preserved; decoding is not recursive.

EXP08 recorded that all 69 stored attack PNGs were recovered exactly and detected for seed 42. Its manifest also records zero score differences across the 755 retained text pairs.

<font color="violet">The original-address score below is saved per source. The corrected candidate verdict is supported by the checked all-69 replay result. No per-candidate R2 score is invented or presented as freshly calculated.</font>

In [ ]:
r2_original = seed_validation.loc[attack_id]
assert int(r2_original["prediction"]) == 1
assert attack_id in stored_ids
same_png_hash = hashlib.sha256(attack_png).hexdigest()
display_case("Case 4 — Recorded R2 correction", attack_png, [
    ("Source ID", str(attack_id)), ("Same PNG SHA-256", same_png_hash),
    ("Model evidence", f"R2 EXP08 seed {SEED}"),
    ("Original-address saved score", f"{float(r2_original['score']):.9f}"),
    ("R2 saved threshold", f"{float(r2_original['threshold']):.9f}"),
    ("Stored candidate verdict", "Phishing — member of checked all-69 detected replay cohort"),
    ("Candidate R2 numerical score", "Not stored per candidate here; not recalculated"),
    ("Decode now", "Exact candidate recovered" if current_exact else "Recovery not confirmed"),
    ("Evidence scope", "Known-attack regression; no new general robustness claim")])
print("The correction is illustrated with the same saved attack, not a newly generated image.")

#### Output Explanation

The stored attack that bypassed R1 belongs to the cohort detected after correction. A separate renewed path search against R2 examined 755 initially detected sources with 90,088 score evaluations and found no text-level evasion candidates.

These results support resistance to the tested unreserved-path encoding family. They do not establish robustness to arbitrary URL changes, parser differences or unseen attacks.

### Case 5 — An Unreadable QR Must Trigger Abstention

We blur the selected clean phishing PNG with sigma 1.0, encode the result as PNG bytes, reload those bytes through the decoder and report its current outcome. If no unique payload is recovered, the pipeline must abstain without invoking a classifier.

The earlier R1 transformation study observed zero coverage for this setting. This one-image check illustrates decoder availability; it does not replace a complete transformed-image evaluation of R2.

In [ ]:
image = cv2.imdecode(np.frombuffer(phishing_png, np.uint8), cv2.IMREAD_GRAYSCALE)
# Match the study's float-pixel blur before quantizing the stored PNG.
blurred = cv2.GaussianBlur(image.astype(np.float32) / 255.0, (0, 0), sigmaX=1.0)
blurred_bytes = np.rint(np.clip(blurred, 0, 1) * 255).astype(np.uint8)
ok, buffer = cv2.imencode(".png", blurred_bytes)
assert ok
blur_png = buffer.tobytes()
blur_decode = decode_png(blur_png)
blur_verdict = "Abstention" if not blur_decode["accepted"] else "Readable — classification not recalculated"
display_case("Case 5 — Decoder availability check", blur_png, [
    ("Source ID", str(int(phishing_example["image_index"]))),
    ("Transformation", "Gaussian blur sigma 1.0; PNG quantization and reload"),
    ("Current payload accepted", str(blur_decode["accepted"])),
    ("Current decoder disagreement", str(blur_decode["conflict"])),
    ("Current pipeline outcome", blur_verdict),
    ("Classifier score", "Not computed"),
    ("Interpretation", "Abstention is neither legitimate classification nor phishing detection")], "violet")
print("The observed decoder outcome is displayed without forcing the historical result.")

#### Output Explanation

If decoding fails, abstention is the correct response to missing input, but phishing protection is unavailable for that image. Zero false alarms under zero coverage is not a robustness success.

If this environment decodes the image, the notebook reports that observation. We do not force an abstention or attach an old score to a different decoded input.

### 4. Put the Selected Examples Back in Context

We return to the complete saved validation results and domain-cluster confidence intervals. Demonstration examples were selected to explain behaviour, whereas these aggregate results summarize all validation sources in the saved run.

In [ ]:
summary = training.loc[training["subset"].eq("Original validation"), [
    "seed", "threshold", "phishing", "phishing_detected", "phishing_missed",
    "phishing_recall", "legitimate", "false_alarms", "false_positive_rate"]]
display(summary.round(6))
display(ci.loc[ci["seed"].eq(SEED), ["metric", "estimate", "CI_95_lower", "CI_95_upper"]].round(6))
print("Previously inspected validation data; confidence intervals condition on fixed fitted models and thresholds.")

#### Output Explanation

All three seeds recorded **870/972 phishing detections (89.51% recall)** and **28/984 false alarms (2.85% URL false-positive rate)**. The 95% domain-cluster intervals were **85.38–93.05%** for recall and **1.76–4.00%** for false positives.

Training seeds used the same fitting, calibration and validation membership. The intervals do not cover alternative splits, refitting, recalibration or bias from repeated development-data inspection. Historical test-set use elsewhere in the research notebook also prevents describing the whole study as untouched-test evaluation.

### 5. Rehearse the Demonstration and Check Readiness

The final checks confirm that the selected stored files can be read and that the current decoder recovers the expected candidate. They do not certify the autonomous deployment package. QA01 will verify the final revision outside notebook-defined functions and in a fresh environment.

In [ ]:
readiness = pd.DataFrame([
    {"check": "EXP08 evidence hashes verified", "passed": True},
    {"check": "Case 1 exact reference recovered now", "passed": legitimate_decode["exact_reference"]},
    {"check": "Case 2 exact reference recovered now", "passed": phishing_decode["exact_reference"]},
    {"check": "Case 3/4 exact candidate recovered now", "passed": bool(current_exact)},
    {"check": "Case 4 belongs to checked all-detected replay cohort", "passed": attack_id in stored_ids},
    {"check": "Case 5 demonstrates abstention now", "passed": not blur_decode["accepted"]}])
display(readiness)
print("Demonstration checks passed." if readiness["passed"].all()
      else "Review the failed checks before presenting. Do not replace them with historical outcomes.")

## Demonstration Conclusion

The examples explain three distinct behaviours: a classifier decision on readable QR content, a representation attack against an earlier model revision, and a decoder abstention when content is unavailable.

Path normalization removed the tested representational difference while the saved clean detection and false-alarm totals remained unchanged. The attack evidence and its correction are preserved as separate results.

<font color="violet">This notebook presents saved model evidence alongside current decoding checks. It makes no fresh classifier-performance claim and does not complete Q14_04_QA01. An autonomous final-model package, fresh-environment reproduction and publication checks remain separate delivery work.</font>

**Presenter sequence:** show cases 1 and 2, compare the same stored attack in cases 3 and 4, explain abstention in case 5, then return to full-cohort metrics and limitations.